# Challenge 12: Comparing SVM, Random Forest, and MLP on Stock Market Movement Prediction

**Author:** Dr Rob Lyon
**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.
This challenge does not constitute financial advice.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
5. [Experimental Setup](#5-experimental-setup)
   - [5.1 Train/Test Split](#51-trainttest-split)
   - [5.2 Defining the Models and Pipelines](#52-defining-the-models-and-pipelines)
6. [Part 1: Cross-Validated Evaluation](#6-part-1-cross-validated-evaluation)
   - [6.1 Running Cross-Validation](#61-running-cross-validation)
   - [6.2 Comparing CV Performance](#62-comparing-cv-performance)
7. [Part 2: Final Model Training and Test Evaluation](#7-part-2-final-model-training-and-test-evaluation)
   - [7.1 Training on the Full Training Set](#71-training-on-the-full-training-set)
   - [7.2 Test Set Evaluation](#72-test-set-evaluation)
   - [7.3 Confusion Matrices](#73-confusion-matrices)
8. [Part 3: ROC Curves](#8-part-3-roc-curves)
   - [8.1 Computing ROC Curves](#81-computing-roc-curves)
   - [8.2 ROC Comparison Plot](#82-roc-comparison-plot)
9. [Part 4: Final Comparison and Judgement](#9-part-4-final-comparison-and-judgement)
   - [9.1 Summary Table](#91-summary-table)
   - [9.2 Reflection Questions](#92-reflection-questions)
10. [Solution](#10-solution)
11. [References](#11-references)

---
## 1. Introduction

Predicting the direction of next-day stock price movement is one of
the most studied problems in quantitative finance and one of the most
contested in academic literature. The efficient market hypothesis holds
that prices already reflect all available public information, making
reliable short-term prediction impossible. In practice, quantitative
traders argue that microstructure patterns, technical indicators, and
sentiment signals carry weak but exploitable predictive content at
short horizons, at least before transaction costs.

Whether or not strong prediction is possible, the problem is an ideal
benchmark for comparing classifiers because:

- The three classes (Down, Flat, Up) are genuinely hard to separate
  with any single algorithm, so real differences between methods emerge
- The feature set mixes correlated technical indicators with relatively
  independent contextual signals, which advantages different algorithms
  in different ways
- The business stakes are asymmetric: a missed Down prediction may mean
  a position is held through a losing day; a false Flat may mean a
  trade opportunity is missed

This challenge takes you through a rigorous multi-classifier comparison
using the **StockMomentum** dataset. You will train an SVM, a Random
Forest, and an MLP on the same data using the same experimental protocol
and compare them on equal terms. The comparison procedure is designed
to eliminate data leakage at every stage: preprocessing parameters are
estimated inside cross-validation folds, the test set is never touched
until the final evaluation, and ROC curves are computed consistently
across all three models.

### Learning Objectives

- Design and implement a fair multi-classifier comparison using
  `Pipeline` to prevent data leakage during cross-validation
- Run stratified k-fold cross-validation for all three models and
  interpret the mean and standard deviation of performance metrics
- Train final models on the full training set and evaluate once on
  a held-out test set
- Compute and compare multi-class ROC curves (one-vs-rest) for all
  three models on the same axes
- Synthesise cross-validation stability, test set accuracy, and ROC
  AUC into a reasoned judgement about which model performs best
  overall and why

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `SVC` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html |
| `RandomForestClassifier` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html |
| `MLPClassifier` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html |
| `Pipeline` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html |
| `cross_validate` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html |
| `roc_curve` and `auc` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html |
| `RocCurveDisplay` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html |
| `label_binarize` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.label_binarize.html |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later and scikit-learn 1.0 or later.
Cross-validation with `n_jobs=-1` runs in parallel. The full cross-
validation pass (three models, 5 folds each) takes approximately 3-8
minutes depending on hardware.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting data |
| `matplotlib` | All plotting |
| `seaborn` | Correlation heatmap |
| `sklearn.pipeline` | `Pipeline` for leak-free preprocessing |
| `sklearn.preprocessing` | `StandardScaler`, `label_binarize` |
| `sklearn.model_selection` | `train_test_split`, `StratifiedKFold`, `cross_validate` |
| `sklearn.svm` | `SVC` |
| `sklearn.ensemble` | `RandomForestClassifier` |
| `sklearn.neural_network` | `MLPClassifier` |
| `sklearn.metrics` | `roc_curve`, `auc`, `classification_report`, `ConfusionMatrixDisplay` |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `market.csv` |
| Location | `data/market.csv` (relative to this notebook) |
| Total rows | 10,000 |
| Features | 12 technical and contextual indicators |
| Target column | `movement` |
| Class 0 | down: ~2,890 samples (~28.9%) |
| Class 1 | flat: ~3,790 samples (~37.9%) |
| Class 2 | up: ~3,320 samples (~33.2%) |

The three-class problem reflects a realistic framing: predicting whether
tomorrow's stock closes more than 0.5% lower (Down), within 0.5% of
today (Flat), or more than 0.5% higher (Up).

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)

CLASS_NAMES  = ['down', 'flat', 'up']
FEATURE_COLS = [
    'rsi_14', 'macd_signal', 'bb_position', 'volume_ratio',
    'atr_pct', 'price_momentum_5d', 'ma_cross', 'overnight_gap',
    'market_sentiment', 'sector_beta', 'earnings_proximity', 'spread_proxy',
]

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/market.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(8)

### 4.2 Understanding the Features

**Technical indicator features:**

| Feature | Range | Description |
|---|---|---|
| `rsi_14` | 0-100 | 14-day Relative Strength Index. >70 = overbought; <30 = oversold. |
| `macd_signal` | ~-2.5 to 2.5 | MACD signal line. Positive = bullish crossover region. Correlated with `ma_cross`. |
| `bb_position` | 0-1 | Position within Bollinger Bands. Near 0 = price at lower band (potential support). |
| `volume_ratio` | 0.2-5.0 | Today's volume / 20-day average. >1 = elevated volume confirms direction. |
| `atr_pct` | 0.1-6.0 | Average True Range as % of price. Volatility measure. Correlated with `spread_proxy`. |
| `price_momentum_5d` | -8 to 8 | 5-day price return %. Momentum signal. Correlated with `rsi_14`. |
| `ma_cross` | -2 to 2 | (5-day MA / 20-day MA) - 1. Positive = short MA above long MA. |
| `overnight_gap` | -3 to 3 | (Today open - yesterday close) / yesterday close %. |

**Contextual features:**

| Feature | Range | Description |
|---|---|---|
| `market_sentiment` | -1 to 1 | Composite sentiment index from news, options, VIX. |
| `sector_beta` | 0.2-2.5 | Stock sensitivity to sector index. |
| `earnings_proximity` | -1 to 1 | Normalised distance to nearest earnings announcement. |
| `spread_proxy` | 0.01-0.25 | Bid-ask spread proxy as % of price. Inversely correlated with `volume_ratio`. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['movement'].value_counts().sort_index()
for k, v in counts.items():
    print(f'  Class {k} ({CLASS_NAMES[k]:6s}): {v:5d}  ({100*v/len(df):.1f}%)')

print()
print('Mean feature values by class:')
print(df.groupby('movement')[FEATURE_COLS].mean().round(3).to_string())

**Questions to consider:**

- The means for `price_momentum_5d` and `ma_cross` differ clearly
  between Down and Up. Do these features show symmetric patterns
  (Down mean = -Up mean)? What does symmetry imply about the
  class structure?
- `rsi_14` has a mean of ~53 for Down and ~46 for Up. Is this the
  expected direction based on the RSI definition (high RSI = overbought)?
- `earnings_proximity` has nearly identical means for all three classes.
  Does this mean it is useless for prediction, or could it still be
  useful in combination with other features?

In [ ]:
# Listing 4.
# Feature distributions for the most informative features
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
colours = {0: 'firebrick', 1: 'steelblue', 2: 'seagreen'}

for ax, feat in zip(axes, FEATURE_COLS[:8]):
    for cls in range(3):
        subset = df.loc[df['movement'] == cls, feat]
        subset.plot(kind='kde', ax=ax, label=CLASS_NAMES[cls],
                    color=colours[cls], linewidth=2)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Technical Feature Distributions by Movement Class (StockMomentum)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- Looking at the KDE plots, how much overlap is there between the Down
  and Up distributions for each feature? Is there a feature where the
  two barely overlap?
- The Flat class (blue) tends to sit between Down and Up for most
  directional features. Does this make intuitive sense?
- Given the substantial overlap between all three classes in most
  features, do you expect any classifier to achieve >75% accuracy?
  What does market efficiency predict?

In [ ]:
# Listing 5.
# Correlation matrix
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.4, ax=ax, square=True,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (StockMomentum)', fontsize=12)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `volume_ratio` and `spread_proxy` are strongly negatively correlated
  (~-0.66). Explain this physically: when volume is high (active market),
  what happens to the bid-ask spread?
- `macd_signal` and `ma_cross` are moderately correlated (~0.46). Both
  measure trend using moving averages. Does having both in the feature
  set add independent information?
- `earnings_proximity` and `sector_beta` are near-zero correlation with
  everything. This means they carry information orthogonal to the
  technical indicators. Which classifier do you expect to exploit
  these independent features most effectively?

---
## 5. Experimental Setup

### 5.1 Train/Test Split

The test set is set aside once and **never used again until Section 8**.
All model selection, hyperparameter choices, and comparisons during
cross-validation are made on the training set only. The test set
provides the final, unbiased estimate of generalisation performance.

After this cell, do not look at or use `X_test` or `y_test` until
Section 8. Every decision made before that section is made without
seeing the test set.

In [ ]:
# Listing 6.
X = df[FEATURE_COLS].values
y = df['movement'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')
print()
print('Class counts in training set:')
for cls in range(3):
    n = int(np.sum(y_train == cls))
    print(f'  {CLASS_NAMES[cls]:6s}: {n:5d}  ({100*n/len(y_train):.1f}%)')

print()
print('The test set is now sealed. It will not be used until Section 8.')

### 5.2 Defining the Models and Pipelines

Each model is wrapped in a `Pipeline` that includes `StandardScaler`
as the first step. This is the correct way to prevent data leakage
during cross-validation: the scaler fits on the training folds inside
each CV split and transforms the validation fold using those parameters.
If you scaled the data before cross-validation, the validation fold
statistics would influence the scaler, which is a form of leakage.

`RandomForestClassifier` does not require scaling (it uses rank-based
splits) but wrapping it in a Pipeline with a scaler does no harm and
makes the experimental setup uniform across all three models.

All three models use `probability=True` (SVC) or support
`predict_proba` natively (RF, MLP), which is required for computing
ROC curves in Section 8.

In [ ]:
# Listing 7.
# TODO: Define the three model pipelines.
#
# Create a dictionary called 'models' with three entries:
#
# 'SVM': Pipeline([
#     ('scaler', StandardScaler()),
#     ('clf', SVC(kernel='rbf', C=1.0, gamma='scale',
#                 probability=True, random_state=42))
# ])
#
# 'Random Forest': Pipeline([
#     ('scaler', StandardScaler()),
#     ('clf', RandomForestClassifier(n_estimators=100,
#                                     random_state=42, n_jobs=-1))
# ])
#
# 'MLP': Pipeline([
#     ('scaler', StandardScaler()),
#     ('clf', MLPClassifier(hidden_layer_sizes=(64, 32),
#                            max_iter=300, random_state=42,
#                            early_stopping=True, validation_fraction=0.1))
# ])
#
# Print the pipeline structure for each model using print(pipeline).

# YOUR CODE HERE

---
## 6. Part 1: Cross-Validated Evaluation

### 6.1 Running Cross-Validation

Use `cross_validate` (not `cross_val_score`) so you can collect
multiple metrics per fold simultaneously. Using `StratifiedKFold`
ensures each fold preserves the class distribution.

Collect two metrics per fold: `accuracy` and `f1_macro`. For a three-
class problem with 30:38:33 distribution, accuracy is a reasonable
headline metric (the class imbalance is mild). Macro-F1 ensures the
Flat class (the hardest to predict) is not hidden by the directional
classes.

**Do not use the test set here.** Pass only `X_train` and `y_train`
to `cross_validate`.

In [ ]:
# Listing 8.
# TODO: Run cross-validation for all three models.
#
# 1. Create StratifiedKFold(n_splits=5, shuffle=True, random_state=42).
#
# 2. For each model in the 'models' dictionary:
#    - Run cross_validate(model, X_train, y_train, cv=cv,
#                          scoring=['accuracy', 'f1_macro'],
#                          n_jobs=-1, return_train_score=True)
#    - Store the result.
#    - Print: model name, mean test accuracy (+/- std),
#                          mean test macro-F1 (+/- std)
#    - Also print mean train accuracy to check for overfitting within CV.
#
# 3. Note: cross_validate with return_train_score=True gives you both
#    training and validation scores for each fold. A large gap between
#    mean train and mean test score within cross-validation indicates
#    the model overfits the training folds.

# YOUR CODE HERE

### 6.2 Comparing CV Performance

In [ ]:
# Listing 9.
# TODO: Visualise the cross-validation results.
#
# Create a figure with two panels (side by side or stacked):
#
# Panel 1: Box plot (or violin plot) of per-fold test ACCURACY for all
#   three models. Each model gets a box showing the distribution of
#   its 5 fold scores.
#   Add a horizontal dashed line at the majority-class baseline
#   (the accuracy of predicting the most common class for everything).
#   Title: 'CV Test Accuracy Distribution (5 folds)'
#
# Panel 2: Box plot of per-fold test MACRO-F1 for all three models.
#   Title: 'CV Test Macro-F1 Distribution (5 folds)'
#
# After plotting, answer in a markdown cell:
#   - Which model has the highest mean CV accuracy?
#   - Which model has the lowest standard deviation across folds?
#     (This measures stability / consistency.)
#   - Is there a clear winner, or are the models within one standard
#     deviation of each other?
#   - Does the train score vs test score gap differ between models?
#     Which model shows the most overfitting?

# YOUR CODE HERE

---
## 7. Part 2: Final Model Training and Test Evaluation

Having compared models using cross-validation, now train each model
on the **full training set** and evaluate once on the **test set**.
This is the final, unbiased estimate of each model's generalisation.

The test set is now unlocked. Evaluate once, report results, and
do not go back to adjust hyperparameters based on what you see.

### 7.1 Training on the Full Training Set

In [ ]:
# Listing 10.
# TODO: Train all three models on the full training set.
#
# For each model in 'models':
#   - Fit on X_train and y_train.
#   - The Pipeline handles scaling internally, so pass raw X_train.
#
# Print a confirmation that each model has been trained.
# Also print the number of support vectors for the SVM
# (access via models['SVM'].named_steps['clf'].n_support_)
# and the feature importances for RF (top 5).

# YOUR CODE HERE

### 7.2 Test Set Evaluation

In [ ]:
# Listing 11.
# TODO: Evaluate all three models on the test set.
#
# For each model:
#   - Predict on X_test (the Pipeline handles scaling).
#   - Compute accuracy, macro-F1.
#   - Print the full classification_report with target_names=CLASS_NAMES.
#
# Create a summary DataFrame with columns:
#   ['Model', 'CV Accuracy', 'CV Macro-F1', 'Test Accuracy', 'Test Macro-F1']
# Fill in the CV values from Section 6 and the test values computed here.
# Print the table.
#
# Questions:
#   - Do the test set results confirm the CV ranking?
#   - Are the test scores close to the CV mean scores? If not, which
#     model shows the largest discrepancy between CV and test?
#   - Which class (down, flat, up) is hardest for each model to predict?

# YOUR CODE HERE

### 7.3 Confusion Matrices

In [ ]:
# Listing 12.
# TODO: Plot confusion matrices for all three models.
#
# Create a figure with 1 row and 3 columns.
# For each model, use ConfusionMatrixDisplay.from_predictions()
# with display_labels=CLASS_NAMES and cmap='Blues'.
# Title each panel with the model name and test accuracy.
#
# Questions from the confusion matrices:
#   - Which model best separates Down from Up (the two directional classes)?
#   - How does each model treat the Flat class?
#     Does any model simply predict Flat for most ambiguous examples?
#   - Are there systematic off-diagonal patterns (e.g. Down predicted
#     as Flat more often than as Up) consistent across all three models?
#     If so, what does this tell you about the feature space structure?

# YOUR CODE HERE

---
## 8. Part 3: ROC Curves

### 8.1 Computing ROC Curves

For a three-class problem, ROC curves use a one-vs-rest (OvR) strategy:
for each class, treat it as the positive class and all others as negative.
This gives three ROC curves per model (one per class), each with its own
AUC. The curves show how well each model discriminates one class from the
rest across all possible classification thresholds.

**Getting probability scores:**
- SVM with `probability=True`: use `predict_proba(X_test)`
- Random Forest: use `predict_proba(X_test)` directly
- MLP: use `predict_proba(X_test)` directly

**Binarising labels for OvR:**
Use `label_binarize(y_test, classes=[0, 1, 2])` to create a binary
indicator matrix of shape (n_samples, 3).

For each model and each class, compute `roc_curve` on the column of the
probability matrix corresponding to that class.

**Important rule**: compute ROC curves on `X_test` and `y_test` only.
These have never been used for training or model selection.

### 8.2 ROC Comparison Plot

In [ ]:
# Listing 13.
# TODO: Compute and plot ROC curves for all three models.
#
# Structure the plot as a 1x3 grid of panels, one per class:
#   Panel 0 (left):   ROC curves for the 'down' class (class 0 vs rest)
#   Panel 1 (centre): ROC curves for the 'flat' class (class 1 vs rest)
#   Panel 2 (right):  ROC curves for the 'up' class (class 2 vs rest)
#
# In each panel, plot one ROC curve per model (3 curves total per panel).
# Use different colours for each model and include the AUC in the legend:
#   e.g. 'SVM (AUC = 0.73)'
# Add the diagonal random-chance line in grey.
# Label axes: 'False Positive Rate' and 'True Positive Rate'.
# Title each panel: 'ROC: {class_name} vs Rest'.
#
# Steps:
# 1. Binarise y_test:
#      y_bin = label_binarize(y_test, classes=[0, 1, 2])
#
# 2. For each model, get probability scores:
#      y_score = model.predict_proba(X_test)  # shape (n_test, 3)
#
# 3. For each class i (0, 1, 2):
#      fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
#      roc_auc = auc(fpr, tpr)
#
# 4. Plot as described above.
#
# After plotting, read the AUC values and answer:
#   - Which class is easiest to distinguish from the rest (highest AUC)?
#   - Which class is hardest (lowest AUC)?
#   - Does the ordering of models (SVM vs RF vs MLP) change between classes?
#   - Is there a model that consistently dominates across all three classes?

# YOUR CODE HERE

---
## 9. Part 4: Final Comparison and Judgement

### 9.1 Summary Table

In [ ]:
# Listing 14.
# TODO: Create a comprehensive summary table.
#
# Build a pandas DataFrame with one row per model and columns:
#   - Model name
#   - CV Accuracy (mean +/- std, formatted as string e.g. '0.654 +/- 0.009')
#   - CV Macro-F1 (mean +/- std)
#   - Test Accuracy
#   - Test Macro-F1
#   - AUC Down (ROC AUC for the down class)
#   - AUC Flat (ROC AUC for the flat class)
#   - AUC Up   (ROC AUC for the up class)
#   - Mean AUC (average of the three OvR AUCs)
#
# Print the table. It should tell a clear, complete story about which
# model is best and on what dimensions.

# YOUR CODE HERE

### 9.2 Reflection Questions

1. Looking at the summary table, is there one model that is clearly best
   on all metrics? If not, which metric should take priority for a
   trading application where missing a Down movement (holding a losing
   position overnight) is more costly than missing an Up movement?

2. The CV standard deviation measures how consistently each model
   performs across different subsets of the training data. A model with
   high mean but high standard deviation may be unstable: it performs
   well on some data splits and poorly on others. Which model would you
   trust most if you had to deploy it without further tuning?

3. The Flat class typically has the lowest ROC AUC of the three.
   Explain why predicting a "no movement" day is harder than predicting
   a directional day, even when directional signals are available.

4. The `Pipeline` structure prevented data leakage during cross-validation
   by fitting the scaler inside each fold. Describe concretely what
   would have gone wrong if you had fitted the scaler on all of `X_train`
   before running cross-validation. Would the effect be large or small
   for this specific dataset, and why?

5. The Random Forest does not require `StandardScaler` because it uses
   rank-based splits. Yet it was wrapped in a Pipeline with a scaler
   in this challenge. Was this necessary? What would change if you
   removed the scaler from the RF pipeline? (Test it if you have time.)

---
## 10. Solution

**Read this section only after attempting the challenge yourself.**

### Step 1: Model Pipelines

In [ ]:
# Listing 15.
models = {
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, gamma='scale',
                    probability=True, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100,
                                        random_state=42, n_jobs=-1))
    ]),
    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(64, 32),
                               max_iter=300, random_state=42,
                               early_stopping=True, validation_fraction=0.1))
    ]),
}

for name, pipe in models.items():
    print(f'{name}:')
    for step_name, step in pipe.steps:
        print(f'  {step_name}: {step.__class__.__name__}')
    print()

**Why Pipelines prevent leakage:**

When `cross_validate` splits the data into training and validation folds,
the Pipeline's `fit_transform` is called on the training fold only.
The validation fold is transformed using the parameters learned from
the training fold. This means the mean and standard deviation used for
scaling are never computed from the validation data. Without a Pipeline,
fitting the scaler on `X_train` before cross-validation would allow the
validation fold's statistics to influence the scaler, which is a subtle
form of data leakage. The effect is usually small (because the full
training set statistics are close to any 80% fold's statistics) but
it is methodologically incorrect and can inflate results on small datasets.

### Step 2: Cross-Validation

In [ ]:
# Listing 16.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    result = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring=['accuracy', 'f1_macro'],
        n_jobs=-1,
        return_train_score=True,
    )
    cv_results[name] = result

    mean_tr_acc = result['train_accuracy'].mean()
    mean_te_acc = result['test_accuracy'].mean()
    std_te_acc  = result['test_accuracy'].std()
    mean_te_f1  = result['test_f1_macro'].mean()
    std_te_f1   = result['test_f1_macro'].std()

    print(f'{name}:')
    print(f'  Train accuracy (mean): {mean_tr_acc:.4f}')
    print(f'  Test  accuracy:        {mean_te_acc:.4f} (+/- {std_te_acc:.4f})')
    print(f'  Test  macro-F1:        {mean_te_f1:.4f} (+/- {std_te_f1:.4f})')
    print(f'  Overfitting gap:       {mean_tr_acc - mean_te_acc:.4f}')
    print()

**Reading the cross-validation results:**

All three models achieve roughly similar test accuracy (within a few
percentage points), which is consistent with the difficult nature of the
prediction problem. The relative ordering will typically be:
SVM ≈ MLP > Random Forest on macro-F1.

The MLP may show a slightly larger train-test gap than the other two
(some overfitting within folds) because its neural architecture has
more parameters relative to the data. The SVM with C=1.0 is
well-regularised and shows a small gap. The Random Forest, while
stable, often under-performs on this problem because the feature
interactions are subtle enough that individual tree splits miss them.

The standard deviation across folds (the ±value) is the consistency
measure: a model with higher mean but higher std is less reliable.
For a deployed trading system, consistency matters as much as peak
performance.

In [ ]:
# Listing 17.
# CV box plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy
acc_data = [cv_results[name]['test_accuracy'] for name in models]
ax1.boxplot(acc_data, labels=list(models.keys()), patch_artist=True,
            boxprops=dict(facecolor='lightsteelblue'),
            medianprops=dict(color='navy', linewidth=2))
majority_baseline = max(np.bincount(y_train)) / len(y_train)
ax1.axhline(majority_baseline, color='firebrick', linestyle='--',
            linewidth=1.5, label=f'Majority baseline ({majority_baseline:.3f})')
ax1.set_ylabel('Test Accuracy', fontsize=11)
ax1.set_title('CV Test Accuracy (5 folds)', fontsize=12)
ax1.legend(fontsize=9)
ax1.set_ylim(0.55, 0.80)

# Macro-F1
f1_data = [cv_results[name]['test_f1_macro'] for name in models]
ax2.boxplot(f1_data, labels=list(models.keys()), patch_artist=True,
            boxprops=dict(facecolor='lightgreen'),
            medianprops=dict(color='darkgreen', linewidth=2))
ax2.set_ylabel('Test Macro-F1', fontsize=11)
ax2.set_title('CV Test Macro-F1 (5 folds)', fontsize=12)
ax2.set_ylim(0.55, 0.80)

plt.suptitle('Cross-Validation Performance: StockMomentum', fontsize=13)
plt.tight_layout()
plt.show()

### Step 3: Train Final Models and Evaluate on Test Set

In [ ]:
# Listing 18.
# Train all models on full training set
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'{name}: trained.')

# SVM support vectors
nsv = sum(models['SVM'].named_steps['clf'].n_support_)
print(f'\nSVM support vectors: {nsv} / {len(X_train)} training samples '
      f'({100*nsv/len(X_train):.1f}%)')

# RF feature importances
rf_imp = pd.Series(
    models['Random Forest'].named_steps['clf'].feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)
print('\nTop 5 Random Forest feature importances:')
print(rf_imp.head(5).round(4).to_string())

In [ ]:
# Listing 19.
# Test set evaluation
test_results = {}
print('=== TEST SET RESULTS ===\n')

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    test_results[name] = {'accuracy': acc, 'f1_macro': f1, 'y_pred': y_pred}
    print(f'--- {name} ---')
    print(f'Accuracy: {acc:.4f}   Macro-F1: {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

### Step 4: Confusion Matrices

In [ ]:
# Listing 20.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, models.items()):
    yp  = test_results[name]['y_pred']
    acc = test_results[name]['accuracy']
    ConfusionMatrixDisplay.from_predictions(
        y_test, yp, display_labels=CLASS_NAMES,
        cmap='Blues', ax=ax, colorbar=False
    )
    ax.set_title(f'{name}\n(test acc = {acc:.3f})', fontsize=10)

plt.suptitle('Test Set Confusion Matrices: StockMomentum', fontsize=13)
plt.tight_layout()
plt.show()

**Reading the confusion matrices:**

The Flat class is typically the most confused: it is the class with the
weakest directional signal, and all three models tend to misclassify
some Flat observations as Down or Up. This is expected: a day that
ends roughly where it started may have looked like it was going
somewhere during the trading session.

Down and Up are more often confused with Flat than with each other,
which makes physical sense: a day that was expected to rise but ended
flat is less unusual than a day that reversed completely. The confusion
matrix off-diagonals should be asymmetric: Down mis-labelled as Flat
is more common than Down mis-labelled as Up.

### Step 5: ROC Curves

In [ ]:
# Listing 21.
# Compute ROC curves for all models and all classes
y_bin = label_binarize(y_test, classes=[0, 1, 2])

roc_data = {}
for name, model in models.items():
    y_score = model.predict_proba(X_test)   # shape (n_test, 3)
    roc_data[name] = {}
    for i, cls_name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc     = auc(fpr, tpr)
        roc_data[name][cls_name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}

In [ ]:
# Listing 22.
# ROC comparison plot
model_colours = {'SVM': 'steelblue', 'Random Forest': 'seagreen', 'MLP': 'darkorange'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, cls_name in zip(axes, CLASS_NAMES):
    for name in models:
        d   = roc_data[name][cls_name]
        ax.plot(d['fpr'], d['tpr'], linewidth=2,
                color=model_colours[name],
                label=f"{name} (AUC = {d['auc']:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random chance')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f"ROC: '{cls_name}' vs Rest", fontsize=12)
    ax.legend(fontsize=9, loc='lower right')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('ROC Curves: SVM vs Random Forest vs MLP (StockMomentum)', fontsize=13)
plt.tight_layout()
plt.show()

**Reading the ROC curves:**

The Down and Up classes typically achieve the highest AUC because they
have the most distinctive feature signatures (clear momentum and MACD
directions). The Flat class has the lowest AUC because its feature
values overlap almost entirely with the borderline regions of Down and Up.

A perfect classifier would have AUC = 1.0: the curve passes through
the top-left corner. A random classifier has AUC = 0.5 (the diagonal).
AUC values in the range 0.68-0.78 for the directional classes and
0.60-0.68 for Flat are consistent with the difficult prediction task
and the market efficiency constraints built into the dataset.

The relative ranking of models on the ROC curves may differ from the
ranking on accuracy or F1. A model may have higher AUC but lower
accuracy if it is better calibrated (its probability scores are more
informative) but uses a suboptimal threshold for hard classification.
This distinction matters in practice: for a trading system that can
act on probability scores rather than hard predictions (e.g. sizing
positions proportional to confidence), a model with higher AUC may
be more valuable even if its accuracy is lower.

### Step 6: Summary Table

In [ ]:
# Listing 23.
summary_rows = []
for name in models:
    cv_acc_mean = cv_results[name]['test_accuracy'].mean()
    cv_acc_std  = cv_results[name]['test_accuracy'].std()
    cv_f1_mean  = cv_results[name]['test_f1_macro'].mean()
    cv_f1_std   = cv_results[name]['test_f1_macro'].std()
    te_acc      = test_results[name]['accuracy']
    te_f1       = test_results[name]['f1_macro']

    auc_down = roc_data[name]['down']['auc']
    auc_flat = roc_data[name]['flat']['auc']
    auc_up   = roc_data[name]['up']['auc']
    mean_auc = np.mean([auc_down, auc_flat, auc_up])

    summary_rows.append({
        'Model':          name,
        'CV Accuracy':    f'{cv_acc_mean:.4f} +/- {cv_acc_std:.4f}',
        'CV Macro-F1':    f'{cv_f1_mean:.4f} +/- {cv_f1_std:.4f}',
        'Test Accuracy':  round(te_acc, 4),
        'Test Macro-F1':  round(te_f1, 4),
        'AUC Down':       round(auc_down, 4),
        'AUC Flat':       round(auc_flat, 4),
        'AUC Up':         round(auc_up, 4),
        'Mean AUC':       round(mean_auc, 4),
    })

summary_df = pd.DataFrame(summary_rows)
print('Final Model Comparison Summary:')
print(summary_df.to_string(index=False))

**Synthesising the results:**

The comparison table brings together all the evidence. When judging
which model is best, consider each dimension:

- **CV accuracy and F1**: the most reliable estimate of generalisation
  because it was computed on data not used for training or selection.
  Prefer the model with the highest mean AND lowest standard deviation.

- **Test accuracy and F1**: confirms whether the CV estimate held up
  on completely unseen data. A large drop from CV to test suggests the
  CV estimate was slightly optimistic (common with small datasets).

- **Mean AUC**: measures discriminative ability across all thresholds.
  A model with higher mean AUC is better at ranking observations by
  predicted probability, which is valuable if the downstream application
  uses probability scores rather than hard labels.

For this problem, the SVM and MLP tend to perform similarly overall,
with SVM often having the edge on consistency (lower CV std). The
Random Forest typically has slightly lower accuracy but comparable AUC,
and its feature importances provide useful interpretability.

The honest conclusion: no model is dramatically better than the others
on a genuinely difficult prediction problem. The right choice depends
on what you optimise for: if you need probability scores (for position
sizing), prefer the model with the best mean AUC. If you need hard
directional predictions, prefer the model with the best CV macro-F1
and lowest CV std. If interpretability matters (which technical
indicators are actually driving predictions), the Random Forest's
feature importances are the most direct answer.

---
## 11. References

1. Fama, E.F. (1970). Efficient capital markets: A review of theory
   and empirical work. *Journal of Finance*, 25(2), 383-417.
   The foundational paper on the efficient market hypothesis that
   motivates the difficulty of the prediction task in this challenge.

2. Jiang, M., Liu, J., Zhang, L., and Liu, C. (2020).
   An improved Stacking framework for stock index prediction by
   leveraging tree-based ensemble models and deep learning algorithms.
   *Physica A: Statistical Mechanics and its Applications*, 541, 122272.
   Example of multi-classifier comparison on stock prediction tasks,
   using the same evaluation framework (CV + ROC) as this challenge.

3. Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5-32.
   https://doi.org/10.1023/A:1010933404324

4. Cortes, C. and Vapnik, V. (1995). Support-vector networks.
   *Machine Learning*, 20(3), 273-297.
   https://doi.org/10.1007/BF00994018

5. Rumelhart, D.E., Hinton, G.E., and Williams, R.J. (1986). Learning
   representations by back-propagating errors.
   *Nature*, 323(6088), 533-536.
   https://doi.org/10.1038/323533a0
   Foundational backpropagation paper underlying MLPClassifier.

6. Fawcett, T. (2006). An introduction to ROC analysis.
   *Pattern Recognition Letters*, 27(8), 861-874.
   https://doi.org/10.1016/j.patrec.2005.10.010
   The standard reference for ROC curve interpretation, including
   multi-class extension via one-vs-rest.

7. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html